# Efficient Attention Benchmark

This notebook benchmarks three attention implementations:
- **Full O(n²):** Baseline — allocates the full n×n score matrix.
- **Sparse (Top-k):** True O(n·k) value aggregation via `torch.gather` — the full n×n value tensor is **never** allocated.
- **Local (Sliding-Window):** True O(n·w) — uses `F.unfold` so only the local neighbourhood is in memory.

We measure both **wall-clock time** and **peak GPU memory** across sequence lengths from 64 to 8192.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from attention_mechanisms import (
    FullAttention, SparseAttention, LocalAttention,
    benchmark_attention, plot_benchmark_results, plot_attention_heatmap
)

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
D_MODEL    = 256
N_HEADS    = 8
BATCH_SIZE = 2
K          = 64
W          = 64

print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Performance & Memory Benchmark

Sequence lengths go from 64 to 8192. At n=8192, Full attention allocates an 8192x8192 score matrix per head — you will see it run out of memory or become dramatically slower. Sparse and Local stay bounded.

In [ ]:
SEQ_LENS = [64, 128, 256, 512, 1024, 2048, 4096, 8192]

models = {
    'Full O(n2)':    FullAttention(D_MODEL, N_HEADS),
    'Sparse (k=64)': SparseAttention(D_MODEL, N_HEADS, k=K),
    'Local (w=64)':  LocalAttention(D_MODEL, N_HEADS, window_size=W),
}

results = {name: [] for name in models}

for name, model in models.items():
    print(f"\nBenchmarking: {name}")
    for n in SEQ_LENS:
        try:
            stats = benchmark_attention(
                model, seq_len=n, d_model=D_MODEL,
                batch_size=BATCH_SIZE, n_runs=10, device=DEVICE
            )
            results[name].append((n, stats['time_ms'], stats['memory_mb']))
            print(f"  n={n:5d} | {stats['time_ms']:8.2f} ms | {stats['memory_mb']:8.1f} MB")
        except torch.cuda.OutOfMemoryError:
            print(f"  n={n:5d} | OOM — full attention cannot fit this sequence length")
            break

print("\nBenchmark complete.")

In [ ]:
plot_benchmark_results(results)

### Reading the plots

**Time plot (left):** On a log-log scale, the slope reveals the complexity class.
- Full O(n2) has slope ~2 — every time n doubles, time quadruples.
- Sparse and Local have slope ~1 — every time n doubles, time only doubles.

**Memory plot (right):** This is where the true implementation difference is clearest.
- Full attention: peak memory grows as n2, you will see it hit OOM at large n.
- Sparse (k=64) and Local (w=64): memory grows as n*k and n*w respectively — the full n*n value tensor is **never** allocated.

## 2. Correctness Check — Attention Heatmaps

Visual inspection confirms each mechanism attends where it should.

In [ ]:
torch.manual_seed(42)
SEQ_VIZ = 128
x_viz = torch.rand(1, SEQ_VIZ, D_MODEL, device=DEVICE)

full_model   = FullAttention(D_MODEL, N_HEADS).to(DEVICE).eval()
sparse_model = SparseAttention(D_MODEL, N_HEADS, k=16).to(DEVICE).eval()
local_model  = LocalAttention(D_MODEL, N_HEADS, window_size=32).to(DEVICE).eval()

with torch.no_grad():
    _, full_w   = full_model(x_viz)
    _, sparse_w = sparse_model(x_viz)
    _, local_w  = local_model(x_viz)

In [ ]:
plot_attention_heatmap(full_w, title="Full Attention — every token attends to every other")

In [ ]:
# Sparse weights are [B, H, n, k] — each row shows the k positions attended to
plot_attention_heatmap(sparse_w, title="Sparse Attention (k=16) — each query attends to only 16 keys\n(x-axis = rank of attended position, not absolute position)")

In [ ]:
# Local weights are [B, H, n, w] — each row shows the w-sized window
plot_attention_heatmap(local_w, title="Local Attention (w=32) — each query attends to its 32-token window\n(x-axis = offset within window, not absolute position)")

### Heatmap interpretation

**Full:** A dense square — every position attends to all n positions. The O(n2) pattern.

**Sparse:** Each row has exactly k=16 attention weights (the top-k selected keys). The x-axis is the rank index within those k positions, not the absolute sequence position.

**Local:** Each row has exactly w=32 attention weights (the local window). The x-axis is the offset within the window (0 to 31). No attention outside the window is allocated at all.

## 3. Complexity Summary Table

In [ ]:
print(f"{'Mechanism':<25} {'Time':<15} {'Memory':<15} {'Key Constraint'}")
print("-" * 75)
rows = [
    ("Full O(n2)",     "O(n2*d)",  "O(n2)",    "Allocates full score matrix"),
    ("Sparse (Top-k)", "O(n*k*d)", "O(n*k*d)", "Value aggregation only; scoring still O(n2)"),
    ("Local (Window)", "O(n*w*d)", "O(n*w*d)", "True O(n*w) via F.unfold — no n*n ever"),
]
for r in rows:
    print(f"{r[0]:<25} {r[1]:<15} {r[2]:<15} {r[3]}")